# 🚀 Nemotron-3-Nano Fine-tuning with NeMo AutoModel (BIRD SQL)

This notebook walks you through how to perform **Low Rank Adapation** (LoRA), a form of **Parameter Efficient Fine Tuning** (PEFT) of the NVIDIA Nemotron-3-Nano-30B model on the **BIRD SQL** text-to-SQL dataset using **NeMo AutoModel**. LoRA trains only a small set of adapter weights instead of the full model, so fine-tuning is faster and uses less memory while still adapting the model to your task.

You will use **two containers**: one to run this notebook for data prep, fine-tuning, merging the adapter into the base model, and testing. and a second container to deploy the merged model via **NIM** so you can send inference requests from this notebook.

---

## 📋 What You're Working With

| | |
|:--:|:--|
| 🤖 **Model** | `NVIDIA-Nemotron-3-Nano-30B-A3B-BF16` |
| 🛠️ **Framework** | NeMo AutoModel (Hugging Face–native) |
| 📐 **Method** | LoRA (Parameter-Efficient Fine-Tuning) |
| 📊 **Dataset** | BIRD SQL with schema (`xu3kev/BIRD-SQL-data-train` via `bird_sql.DatasetBIRD`) |

---

## ✅ Prerequisites

**Hardware:** 2× GPUs (e.g. H100, A100), sufficient storage for model and checkpoints.  
**Software:** Linux, Docker, NVIDIA Container Toolkit. Run this notebook **inside** the NeMo AutoModel container; then register and select the **Python (AutoModel venv)** kernel (Step 1).

---

## 🗺️ Workflow at a Glance

| Step | What you'll do |
|:--:|:--|
| **1** | 🐳 Pull and run the NeMo AutoModel container (mount this repo); register and select **Python (AutoModel venv)** kernel for Jupyter |
| **2** | 📊 Prepare BIRD SQL dataset (train → JSONL) |
| **3** | ⚙️ Configure PEFT training (YAML) |
| **4** | 🎯 Run fine-tuning inside the container |
| **5** | 🔀 Merge LoRA adapter into base model → `merged_model/` for NIM |
| **6** | Two-container setup: Docker network, spin up NIM with merged model |
| **7** | Inference on NIM: one request from the first training example to confirm NIM works |

---

## Step 1: Run with the NeMo AutoModel Container

We highly recommend that you run this tutorial inside the official [NeMo AutoModel container](https://catalog.ngc.nvidia.com/orgs/nvidia/containers/nemo-automodel). This image is specially configured to run NeMo AutoModel, PyTorch, and all of their dependencies for fine-tuning Nemotron-3-Nano.

### Launch the container (on your host)

Run these **on your host**. They create the **evalnet** network, mount your repo at `/workspace`, and start the notebook container. Set `REPO_PATH` to the path that contains `usage-cookbook`.

```bash
docker network create evalnet
export REPO_PATH=/path/to/30b-bird
docker pull nvcr.io/nvidia/nemo-automodel:25.11.00
docker run -it --gpus all -p 8889:8888 --network evalnet --name notebook \
  -v "${REPO_PATH}:/workspace" nvcr.io/nvidia/nemo-automodel:25.11.00 bash
```

If the container is already running, attach with `docker exec -it notebook bash` instead of `docker run`. 


### Start Jupyter (inside the container)

Inside the container run the following:

```bash
cd /workspace/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment
jupyter notebook --ip=0.0.0.0 --allow-root
```

From your machine: forward port **8889** (e.g. `ssh -L 8889:localhost:8889 user@host`), then open **http://localhost:8889** and paste the token Jupyter printed.


### Connect Jupyter to the AutoModel venv kernel 

Register the venv so the notebook has `nemo_automodel`.

In a terminal (inside the container):

```bash
source /opt/venv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name automodel-venv --display-name "Python (AutoModel venv)"
```

Then in restart Jupyter and in the notebook, select **Kernel → Change kernel → Python (AutoModel venv)** so the notebook uses the correct environment.

In [ ]:
# Verify environment (run inside the container):
import sys
try:
    import nemo_automodel
    import datasets
    print("nemo_automodel:", nemo_automodel.__file__)
    print("datasets OK")
except ImportError as e:
    print("ImportError (run this notebook inside the container):", e)
try:
    import mamba_ssm
    print("mamba_ssm OK")
except ImportError as e:
    print("mamba_ssm:", e)


**If the cell above reports `No module named 'nemo_automodel'`:** Usually the notebook is using the wrong kernel; switch to **Kernel → Python (AutoModel venv)**.

### Accessing Jupyter from your browser

If you started the container with **port 8889** mapped (e.g. `-p 8889:8888`):

1. **Make port 8889 reachable from your machine.** If the host is remote, forward that port (e.g. SSH: `ssh -L 8889:localhost:8889 user@host`, or your IDE's port-forwarding / "Ports" feature). If you're on the same machine as the host, you can use `localhost` directly.
2. **Get the token from the container.** In the terminal where Jupyter is running, it prints a URL with a token (e.g. `http://127.0.0.1:8888/tree?token=...`). Copy only the **token** (the long string after `token=`).
3. **Open Jupyter in your browser.** Go to **http://localhost:8889/tree?token=YOUR_TOKEN** (replace `YOUR_TOKEN` with the token from step 2). Use **8889** (the host port you mapped), not 8888 (the port inside the container).
4. **Port 8889 is for the container.** If the host also runs Jupyter on 8888, that's separate; the container's Jupyter is on 8889.

## Step 2: Prepare the BIRD SQL Dataset

**`bird_sql.DatasetBIRD`** loads BIRD from Hugging Face with schema per example; prompts use the Nemotron chat template. The cell below writes **JSONL** for AutoModel.


In [ ]:
import os
import sys
from datasets import disable_caching

disable_caching()

# Ensure bird_sql is importable (cwd or repo-relative path)
cwd = os.getcwd()
for candidate in [cwd, os.path.join(cwd, "usage-cookbook", "Nemotron-3-Nano", "finetuning_and_deployment")]:
    bird_sql_dir = os.path.join(candidate, "bird_sql")
    if os.path.isdir(bird_sql_dir):
        if candidate not in sys.path:
            sys.path.insert(0, candidate)
        break
from bird_sql.dataset_bird import DatasetBIRD

DATASET_DIR = os.environ.get("DATASET_DIR", os.path.join(os.getcwd(), "dataset"))
os.makedirs(DATASET_DIR, exist_ok=True)
training_jsonl = os.path.join(DATASET_DIR, "training.jsonl")

model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
max_seq_len = 4096
num_workers = 4

print("Preparing BIRD training dataset (with schema) via bird_sql.DatasetBIRD...")
dataset = DatasetBIRD(
    model_id_to_prep_for=model_id,
    max_seq_len=max_seq_len,
    num_workers=num_workers,
).make_dataset()
dataset = dataset.sort("length")
# Save only input/output for AutoModel JSONL (same shape as before)
dataset = dataset.select_columns(["input", "output"])
dataset.to_json(training_jsonl, orient="records", lines=True, force_ascii=False)
print(f"Saved {len(dataset)} examples to {training_jsonl}")

## Step 3: PEFT Training Config (YAML)

Create a YAML config that points the AutoModel PEFT recipe at the Nemotron-3-Nano model and your BIRD JSONL. The dataset is loaded via **ColumnMappedTextInstructionDataset** with `input` → question and `output` → answer, and **answer-only loss**.

In [ ]:
import os

DATASET_DIR = os.environ.get("DATASET_DIR", os.path.join(os.getcwd(), "dataset"))
training_jsonl = os.path.join(DATASET_DIR, "training.jsonl")
config_path = os.path.join(os.getcwd(), "bird_peft_nemotron_nano.yaml")

yaml_content = f"""# PEFT (LoRA) fine-tuning: Nemotron-3-Nano on BIRD SQL (2 GPUs)
# Run from this directory inside the container: torchrun --nproc-per-node=2 finetune.py

step_scheduler:
  global_batch_size: 8
  local_batch_size: 4
  ckpt_every_steps: 200
  val_every_steps: 200
  max_steps: 1000
  num_epochs: 1

dist_env:
  backend: nccl
  timeout_minutes: 60

rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1111
  ranked: true

model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  torch_dtype: bf16

checkpoint:
  enabled: true
  checkpoint_dir: checkpoints/
  model_save_format: safetensors
  save_consolidated: false

peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  match_all_linear: true
  dim: 8
  alpha: 32
  use_triton: true

distributed:
  _target_: nemo_automodel.components.distributed.fsdp2.FSDP2Manager
  dp_size: null
  tp_size: 1
  cp_size: 1
  sequence_parallel: false
  backend: nccl

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: dataset/training.jsonl
  # split is ignored for local JSONL; omit or set for HF datasets
  split: train
  column_mapping:
    question: input
    answer: output
  answer_only_loss_mask: true

packed_sequence:
  packed_sequence_size: 0

dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater
  shuffle: false

optimizer:
  _target_: torch.optim.Adam
  betas: [0.9, 0.999]
  eps: 1e-8
  lr: 1.0e-5
  weight_decay: 0
"""

with open(config_path, "w") as f:
    f.write(yaml_content)
print(f"Wrote config to {config_path}")
print(f"Dataset path in config: dataset/training.jsonl")

**Note:** If the dataset step fails on `split`, try removing or setting `split: null` in the YAML.

**Parameters you can change** (in the YAML above or in `bird_peft_nemotron_nano.yaml`):

- **`global_batch_size`** (default 8): Total examples per optimizer step across all GPUs. Larger values give stabler gradients but need more GPU memory. 8 fits on 2 GPUs for this model; reduce if OOM.
- **`local_batch_size`** (default 4): Examples per GPU per step. Must divide `global_batch_size`; typically `global_batch_size / num_gpus`. Set to 4 so each of 2 GPUs gets 4 samples.
- **`ckpt_every_steps`** (default 200): Save a checkpoint every N steps. 200 gives a few checkpoints in a 1000-step run without writing too often.
- **`max_steps`** (default 1000): Stop training after this many steps. 1000 keeps the run to about 1–2 hours; lower for a quicker test, higher for more training.
- **`peft.dim`** (default 8): LoRA rank (width of adapter matrices). **`peft.alpha`** (default 32): LoRA scaling (often 2× or 4× dim). dim=8, alpha=32 is a common balance of capacity vs overfitting and memory.
- **`optimizer.lr`** (default 1e-5): Learning rate. 1e-5 is a typical starting point for LoRA on LLMs; lower for stability, higher for faster (but riskier) convergence.


## Step 4: Run Fine-Tuning

Run the cell below to start PEFT fine-tuning. The number of steps is set in the YAML (Step 3) and depends on dataset size and batch size; expect **about 1–2 hours**. This model needs a minimum amount of GPU memory; if you hit an **out-of-memory** error or want a **shorter run**, edit the YAML in Step 3 (e.g. reduce `global_batch_size` / `local_batch_size` or lower `max_steps`). Checkpoints go to `checkpoints/`. Adjust `--nproc-per-node` in the cell if you have a different GPU count.


In [ ]:
%%bash
# adjust --nproc-per-node to your GPU count (default 2)
torchrun --nproc-per-node=2 finetune.py --config bird_peft_nemotron_nano.yaml


## Step 5: Merge Adapter into Base Model for NIM

NIM expects a **merged** Hugging Face model (not LoRA alone). This step loads the **LATEST** checkpoint from `checkpoints/`, merges the adapter with the base model via `peft`, and saves to `merged_model/` for Step 6. Expect a few minutes: it loads the full base model (~30B parameters) and the adapter, merges them in memory, then writes the full merged model to disk.


In [ ]:
# Step 5 — Merge LoRA adapter into base model for NIM (Hugging Face format).
# Uses checkpoints/LATEST (points to the most recently saved checkpoint). 
import os
from pathlib import Path

RECIPE_DIR = Path("/workspace/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment")
CHECKPOINT_PATH = (RECIPE_DIR / "checkpoints" / "LATEST").resolve() / "model"
MERGED_DIR = RECIPE_DIR / "merged_model"

assert CHECKPOINT_PATH.is_dir(), f"Checkpoint dir not found: {CHECKPOINT_PATH} (run Step 4 first and save a checkpoint)."

from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

print("Loading PEFT checkpoint (base model + adapter) from LATEST...")
model = AutoPeftModelForCausalLM.from_pretrained(str(CHECKPOINT_PATH), trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT_PATH))

print("Merging adapter into base model...")
model = model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
print(f"Saving merged model to {MERGED_DIR}...")
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Done. Use merged_model/ as the NIM workspace mount in Step 6.")

## Step 6: Set Up the NIM Container (Merged Model on evalnet)

We deploy our model using a [NIM container](https://catalog.ngc.nvidia.com/orgs/nim/containers/nemotron-3-nano) in order to send inference requests from this notebook.

NIM must run on **evalnet** (the same network as this notebook from Step 1) so the notebook can reach it at `http://nim:8000` in Step 7. 

Copy-paste one of the blocks below into a **terminal on the host**. Set `REPO_PATH` to the host path you used in Step 1 for `-v ${REPO_PATH}:/workspace`. You can obtain an NGC API key from [NGC Catalog](https://catalog.ngc.nvidia.com). Wait for the NIM to finish launching before running Step 7.

**Deploy our fine-tuned merged model:**

```bash
docker network create evalnet 2>/dev/null || true
export REPO_PATH=/path/to/30b-bird
export NGC_API_KEY=your_ngc_key
export MERGED="${REPO_PATH}/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment/merged_model"
docker run -it --rm --gpus all --network evalnet --name nim \
  -e NGC_API_KEY=$NGC_API_KEY \
  -e NIM_DISABLE_MODEL_DOWNLOAD=1 \
  -e NIM_RELAX_MEM_CONSTRAINTS=1 \
  -e NIM_MODEL_PROFILE=7cbe1181600064c6e10ebaf843497acae35aacff2ab96fe8247ae541ae0ac28a \
  -v "${MERGED}:/opt/nim/workspace" \
  -p 8000:8000 \
  nvcr.io/nim/nvidia/nemotron-3-nano:1.7.0-variant
```



## Step 7: Inference on NIM

We send an HTTP request from this notebook to the NIM endpoint (`http://nim:8000`) using the first example from our training data as a smoke test. The cell prints the prompt, expected SQL, and the model's reply. NIM must be running on **evalnet** as `nim` (Step 6).

In [ ]:
# Step 7 — Inference on NIM: one request using the first training example.
import json
import os
import requests

NIM_BASE_URL = "http://nim:8000"
NIM_MODEL_ID = "nvidia/nemotron-3-nano"
NIM_SYSTEM_FOR_SQL = (
    "detailed thinking off. You are a text-to-SQL assistant. "
    "Given a question and evidence about the database, respond with only the SQL query, "
    "no explanation, no markdown, no other text. "
    "Prefer the simplest query that answers the question: do not use table aliases (e.g. T1, T2) or JOINs unless the question truly requires multiple tables; when one table is enough, query it directly."
)
MAX_NEW_TOKENS = 512

# Load first example from training JSONL
training_path = "dataset/training.jsonl" if os.path.exists("dataset/training.jsonl") else "training.jsonl"
if not os.path.exists(training_path):
    raise FileNotFoundError(f"Training file not found: {training_path}. Run Step 2 first.")
with open(training_path) as f:
    first = json.loads(f.readline())
prompt_input = first["input"].strip()
expected = first["output"].strip()

def get_content_from_nim_response(data):
    try:
        choice = data.get("choices", [{}])[0]
        msg = choice.get("message", {})
        content = msg.get("content")
        if isinstance(content, str) and content:
            return content
        rc = msg.get("reasoning_content") or msg.get("reasoning")
        if isinstance(rc, str) and rc.strip():
            return rc.strip()
        return ""
    except Exception:
        return ""

url = f"{NIM_BASE_URL}/v1/chat/completions"
payload = {
    "model": NIM_MODEL_ID,
    "messages": [
        {"role": "system", "content": NIM_SYSTEM_FOR_SQL},
        {"role": "user", "content": prompt_input + "\n\nOutput only the SQL query, nothing else."},
    ],
    "max_tokens": MAX_NEW_TOKENS,
    "temperature": 0,
    "chat_template_kwargs": {"enable_thinking": False},
}

try:
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    content = get_content_from_nim_response(r.json())
    print("--- Input ---")
    print(repr(prompt_input))
    print("\n--- Expected (gold) ---")
    print(repr(expected))
    print("\n--- Model reply ---")
    print(repr(content if content else "(empty)"))
except requests.exceptions.ConnectionError as e:
    print("✗ Connection failed. Is NIM running on evalnet? Start it with --network evalnet --name nim")
    print("  Error:", e)
except Exception as e:
    print("Error:", e)